# 05 - End-to-End Orchestration

This notebook demonstrates the complete workflow: loading invoices, extracting data, validating, and submitting to Fakturoid.

In [ ]:
from src.document_processor import DocumentProcessor
from src.ai_extractor import AIExtractor, InvoiceData
from src.fakturoid_client import FakturoidClient
from src.config import config
from src.agent import InvoiceProcessingAgent
import json
from pathlib import Path
from datetime import datetime

In [ ]:
# Initialize agent
agent = InvoiceProcessingAgent(config)

print("Invoice Processing Agent initialized")
print(f"Mode: {config.processing.mode}")
print(f"Auto-submit: {config.processing.auto_submit}")
print(f"Invoices directory: {config.directories.invoices}")

In [ ]:
# Test connections
print("Testing connections...")
if agent.test_connections():
    print("✓ All connections successful!")
else:
    print("✗ Connection test failed. Check your credentials.")

In [ ]:
# Process all invoices with manual review
print("="*60)
print("PROCESSING ALL INVOICES")
print("="*60)

# Get list of files first
files = agent.doc_processor.list_invoice_files()
print(f"\nFound {len(files)} invoice files")

if len(files) == 0:
    print(f"\nNo invoice files found in: {agent.invoices_dir}")
    print("Please add PDF or image files to process.")
else:
    # Process batch with review enabled
    results = agent.process_batch(review=True, max_files=None)
    
    print(f"\n{'='*60}")
    print("PROCESSING SUMMARY")
    print(f"{'='*60}")
    submitted = sum(1 for r in results if r['status'] == 'submitted')
    extracted = sum(1 for r in results if r['status'] == 'extracted')
    errors = sum(1 for r in results if r['status'] == 'error')
    
    print(f"Total invoices: {len(results)}")
    print(f"Submitted: {submitted}")
    print(f"Extracted (pending review): {extracted}")
    print(f"Failed: {errors}")

In [ ]:
# Show detailed results
if 'results' in locals() and results:
    print("\nDetailed Results:")
    for result in results:
        print(f"\n{'-'*60}")
        print(f"File: {result['file']}")
        print(f"Status: {result['status']}")
        
        if result['status'] == 'submitted':
            if result.get('extracted_data'):
                data = result['extracted_data']
                print(f"Invoice Number: {data.get('invoice_number', 'N/A')}")
                print(f"Supplier: {data.get('supplier_name', 'N/A')}")
                print(f"Amount: {data.get('total_amount', 'N/A')} {data.get('currency', 'CZK')}")
            if result.get('fakturoid_response'):
                print(f"Fakturoid ID: {result['fakturoid_response'].get('id')}")
                print(f"Fakturoid Number: {result['fakturoid_response'].get('number')}")
                print(f"URL: {result['fakturoid_response'].get('html_url')}")
        
        elif result['status'] == 'extracted':
            if result.get('extracted_data'):
                data = result['extracted_data']
                print(f"Invoice Number: {data.get('invoice_number', 'N/A')}")
                print(f"Supplier: {data.get('supplier_name', 'N/A')}")
                print(f"Amount: {data.get('total_amount', 'N/A')} {data.get('currency', 'CZK')}")
                print("⚠️  Awaiting manual review/approval")
        
        elif result['status'] == 'error':
            print(f"Error: {result.get('error', 'Unknown error')}")
else:
    print("No results to display. Run the previous cell first.")

In [ ]:
# Process a single invoice with detailed steps
files = agent.doc_processor.list_invoice_files()

if files:
    test_file = files[0]
    print(f"Processing single invoice: {test_file.name}")
    print("="*60)
    
    # Use the agent's process_file method
    print("\n📄 Processing invoice...")
    result = agent.process_file(test_file, review=True)
    
    print(f"\n{'='*60}")
    print(f"Status: {result['status']}")
    print(f"{'='*60}")
    
    if result['status'] == 'error':
        print(f"\n✗ Error: {result['error']}")
    
    elif result['status'] == 'extracted':
        print("\n✓ Invoice data extracted successfully!")
        print("⚠️  Review the data above. To submit, set auto_submit=True")
    
    elif result['status'] == 'submitted':
        print("\n✓ Invoice submitted to Fakturoid!")
        if result.get('fakturoid_response'):
            resp = result['fakturoid_response']
            print(f"   ID: {resp.get('id')}")
            print(f"   Number: {resp.get('number')}")
            print(f"   URL: {resp.get('html_url')}")
else:
    print("No invoice files found.")
    print(f"Please add PDF or image files to: {agent.invoices_dir}")

In [ ]:
# Check processed invoices directory
processed_dir = config.directories.processed
if processed_dir.exists():
    processed_files = [f for f in processed_dir.glob("*") if f.is_file() and not f.name.startswith('.')]
    
    print(f"Processed files directory: {processed_dir}")
    print(f"Number of processed files: {len(processed_files)}")
    
    if processed_files:
        print("\nProcessed files (most recent first):")
        sorted_files = sorted(processed_files, key=lambda x: x.stat().st_mtime, reverse=True)
        for f in sorted_files[:10]:
            mtime = datetime.fromtimestamp(f.stat().st_mtime)
            print(f"  - {f.name} ({mtime.strftime('%Y-%m-%d %H:%M')})") 
        if len(processed_files) > 10:
            print(f"  ... and {len(processed_files) - 10} more")
    else:
        print("\n📁 No processed files yet")
else:
    print(f"Processed directory doesn't exist yet: {processed_dir}")
    print("It will be created when the first invoice is submitted.")